In [9]:
import xarray as xr
import climpred as cp
from xarray import DataArray

In [10]:
range_id = 0
dataset_name = "ukmo_604"
# Mask and scale is used for scaling factor
dataset = xr.open_dataset(f"/scratch/tm70/ag9761/iceds_datasets/{dataset_name}_{range_id}_new.nc", mask_and_scale=False, chunks="auto")
dataset = xr.decode_cf(dataset)

In [11]:
def convert_units(da: DataArray, new_unit: str) -> DataArray:
        da = da.metpy.quantify()
        da = da.metpy.convert_units(new_unit)
        da = da.metpy.dequantify()
        return da

dataset["tprate"] = convert_units(dataset["tprate"], "mm/day")

In [12]:
# Make dataset ready for climpred
renamed_vars = ["indexing_time", "forecast_reference_time"]
for rv in renamed_vars:
    if rv in dataset.coords:
        dataset = dataset.rename({rv : "init"})
dataset = dataset.rename({"forecastMonth" : "lead", "number": "member", "tprate" : "pr"})
dataset["lead"].attrs["units"] = "months"
dataset = dataset.assign_coords(lead=dataset["lead"] - 1)

In [13]:
# Convert from init + lead config to valid_times for leads
model = cp.HindcastEnsemble(dataset).get_initialized()
dataset = cp.utils.convert_init_lead_to_valid_time_lead(model)
dataset

/scratch/tm70/ag9761/conda/envs/iceds/lib/python3.13/site-packages/climpred/classes.py:1003: UserWarning: HindcastEnsemble is chunked along dimensions ['member', 'init'] with more than one chunk. `HindcastEnsemble.chunks=Frozen({'member': (7, 7, 7, 7), 'init': (72, 72, 143), 'lead': (1, 1, 1, 1, 1, 1), 'latitude': (46, 46, 89), 'longitude': (46, 46, 89)})`.
You cannot call `HindcastEnsemble.verify` or `HindcastEnsemble.bootstrap` in combination with any of  ['member', 'init'] passed as `dim`. In order to do so, please rechunk ['member', 'init'] with `HindcastEnsemble.chunk({{dim:-1}}).verify(dim=dim).`
If you do not want to use dimensions  ['member', 'init'] in `HindcastEnsemble.verify(dim=dim)`, you can disregard this warning.
Consider chunking embarrassingly parallel dimensions such as ['latitude', 'longitude'] automatically, i.e. `HindcastEnsemble.chunk(latitude="auto").verify(...).
  warnings.warn(msg)


<xarray.Dataset> Size: 6GB
Dimensions:     (lead: 6, member: 28, valid_time: 292, latitude: 181,
                 longitude: 181)
Coordinates:
  * lead        (lead) int64 48B 0 1 2 3 4 5
  * member      (member) int64 224B 0 1 2 3 4 5 6 7 ... 20 21 22 23 24 25 26 27
  * valid_time  (valid_time) object 2kB 1993-02-01 00:00:00 ... 2017-05-01 00...
    init        (lead, valid_time) object 14kB 1993-02-01 00:00:00 ... 2016-1...
  * latitude    (latitude) float64 1kB -14.0 -14.05 -14.1 ... -22.9 -22.95 -23.0
  * longitude   (longitude) float64 1kB 140.0 140.1 140.1 ... 148.9 149.0 149.0
Data variables:
    pr          (lead, member, valid_time, latitude, longitude) float32 6GB dask.array<chunksize=(1, 7, 95, 46, 46), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             egrr
    GRIB_centreDescription:  U.K. Met Office - Exeter
    GRIB_subCentre:          98
    Conventions:             CF-1.7
    institution:             U.K. Met Office - Exeter

In [14]:
dataset = dataset.rename({"valid_time" : "time"})
dataset = dataset.drop_vars(["init"])

In [15]:
# Prediction on ensemble median
# NOTE: Having lower prediction value
dataset = dataset.median(dim='member')

In [16]:
dataset.to_netcdf(f"/scratch/tm70/ag9761/iceds_datasets/{dataset_name}_{range_id}_new_processed.nc")

/scratch/tm70/ag9761/conda/envs/iceds/lib/python3.13/site-packages/dask/_task_spec.py:767: RuntimeWarning: All-NaN slice encountered
  return self.func(*new_argspec, **kwargs)


In [ ]:
dataset